<a href="https://colab.research.google.com/github/un1u3/ml-labs/blob/main/Deep%20Learning/Pytorch/10_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# transfer learning is one of the most powwerful techniques in DL
# Instead of training a NN from scratch, we use a pre-trained model(trained on milliions of images)
# and adapt to our specific task.


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import os
import requests
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device : {device}")

device : cuda


In [4]:
# download and prepare data
import torchvision.datasets as datasets
import urllib.request
import tarfile
import shutil
from sklearn.model_selection import train_test_split

def download_oxford_pets():
    """
    Download Oxford-IIIT Pet Dataset (37 categories of pets).
    We'll use just cats vs dogs from this.
    """
    data_dir = Path('data/oxford_pets')
    data_dir.mkdir(parents=True, exist_ok=True)

    print("Downloading Oxford-IIIT Pet Dataset...")

    # Download images
    images_url = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
    images_path = data_dir / "images.tar.gz"

    if not images_path.exists():
        urllib.request.urlretrieve(images_url, images_path)
        print("✓ Images downloaded")

    # Extract
    if not (data_dir / "images").exists():
        with tarfile.open(images_path, 'r:gz') as tar:
            tar.extractall(data_dir)
        print("✓ Images extracted")

    return data_dir / "images"


def organize_pets_dataset(source_dir):
    """
    Organize Oxford pets into cats vs dogs format.
    Pet names starting with capital letters are cats, lowercase are dogs.
    """
    train_dir = Path('data/dogs_vs_cats/train')
    val_dir = Path('data/dogs_vs_cats/val')

    # Create directories
    for split_dir in [train_dir, val_dir]:
        (split_dir / 'cats').mkdir(parents=True, exist_ok=True)
        (split_dir / 'dogs').mkdir(parents=True, exist_ok=True)

    # Get all images
    all_images = list(Path(source_dir).glob('*.jpg'))

    # Separate cats and dogs based on naming convention
    cat_images = []
    dog_images = []

    for img_path in all_images:
        name = img_path.stem
        # Extract breed name (before the number)
        breed = '_'.join(name.split('_')[:-1])

        # Common cat breeds in dataset
        cat_breeds = ['Abyssinian', 'Bengal', 'Birman', 'Bombay', 'British_Shorthair',
                      'Egyptian_Mau', 'Maine_Coon', 'Persian', 'Ragdoll', 'Russian_Blue',
                      'Siamese', 'Sphynx']

        if any(breed.startswith(cat_breed) for cat_breed in cat_breeds):
            cat_images.append(img_path)
        else:
            dog_images.append(img_path)

    # Split into train/val (80/20)
    cat_train, cat_val = train_test_split(cat_images, test_size=0.2, random_state=42)
    dog_train, dog_val = train_test_split(dog_images, test_size=0.2, random_state=42)

    # Copy files
    print(f"Organizing {len(cat_train)} cat training images...")
    for img in cat_train:
        shutil.copy(img, train_dir / 'cats' / img.name)

    print(f"Organizing {len(cat_val)} cat validation images...")
    for img in cat_val:
        shutil.copy(img, val_dir / 'cats' / img.name)

    print(f"Organizing {len(dog_train)} dog training images...")
    for img in dog_train:
        shutil.copy(img, train_dir / 'dogs' / img.name)

    print(f"Organizing {len(dog_val)} dog validation images...")
    for img in dog_val:
        shutil.copy(img, val_dir / 'dogs' / img.name)

    print("\n✓ Dataset organized!")
    print(f"  Train: {len(cat_train)} cats, {len(dog_train)} dogs")
    print(f"  Val: {len(cat_val)} cats, {len(dog_val)} dogs")


def setup_dataset():
    """Main function to download and setup dataset."""

    # Check if already exists
    if (Path('data/dogs_vs_cats/train/cats').exists() and
        len(list(Path('data/dogs_vs_cats/train/cats').glob('*.jpg'))) > 0):
        print("✓ Dataset already exists!")
        return

    print("Setting up dataset...")

    # Download and organize
    source_dir = download_oxford_pets()
    organize_pets_dataset(source_dir)

    print("\n" + "="*60)
    print("Dataset ready for training!")
    print("="*60)

# Download and setup dataset
setup_dataset()

Setting up dataset...
✓ Images downloaded


/tmp/ipython-input-3375940449.py:29: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(data_dir)


✓ Images extracted
Organizing 1920 cat training images...
Organizing 480 cat validation images...
Organizing 3992 dog training images...
Organizing 998 dog validation images...

✓ Dataset organized!
  Train: 1920 cats, 3992 dogs
  Val: 480 cats, 998 dogs

Dataset ready for training!


In [5]:
class DogsVsCatsDataset(Dataset):
    """Custom Dataset for Dogs vs Cats classification."""

    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (string): Directory with all the images (e.g., 'data/train')
            transform (callable, optional): Optional transform to be applied on images
        """
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.classes = ['cats', 'dogs']
        self.class_to_idx = {'cats': 0, 'dogs': 1}

        # Get all image paths and labels
        self.samples = []
        for class_name in self.classes:
            class_dir = self.root_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*.jpg'):
                    self.samples.append((str(img_path), self.class_to_idx[class_name]))

        print(f"Found {len(self.samples)} images in {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [6]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transforms with augmentation
train_transform = transforms.Compose([
    transforms.Resize(256),                      # Resize to 256x256
    transforms.RandomCrop(224),                  # Random crop to 224x224
    transforms.RandomHorizontalFlip(p=0.5),      # Random flip
    transforms.RandomRotation(15),               # Random rotation
    transforms.ColorJitter(brightness=0.2,
                          contrast=0.2,
                          saturation=0.2),       # Color augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN,
                        std=IMAGENET_STD)        # Normalize with ImageNet stats
])

# Validation transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),                  # Center crop only
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN,
                        std=IMAGENET_STD)
])


In [7]:
train_dataset = DogsVsCatsDataset(
    root_dir='data/dogs_vs_cats/train',
    transform=train_transform
)

val_dataset = DogsVsCatsDataset(
    root_dir='data/dogs_vs_cats/val',
    transform=val_transform
)

# Create dataloaders
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Found 5912 images in data/dogs_vs_cats/train
Found 1478 images in data/dogs_vs_cats/val
Training batches: 185
Validation batches: 47


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [8]:
def create_transfer_model(num_classes=2, freeze_backbone=True):

    # Load pre-trained ResNet18
    model = models.resnet18(pretrained=True)

    print("Pre-trained ResNet18 loaded!")
    print(f"Original final layer: {model.fc}")

    # Freeze all layers if specified
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
        print("✓ All backbone layers frozen")

    # Replace the final fully connected layer
    # ResNet18 has 512 features before the final layer
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)

    print(f"New final layer: {model.fc}")
    print(f"✓ Final layer replaced ({num_features} -> {num_classes})")

    return model

# Create transfer learning model
transfer_model = create_transfer_model(num_classes=2, freeze_backbone=True)
transfer_model = transfer_model.to(device)

# Count trainable parameters
trainable_params = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in transfer_model.parameters())

print(f"\nModel Parameters:")
print(f"Total: {total_params:,}")
print(f"Trainable: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 104MB/s]


Pre-trained ResNet18 loaded!
Original final layer: Linear(in_features=512, out_features=1000, bias=True)
✓ All backbone layers frozen
New final layer: Linear(in_features=512, out_features=2, bias=True)
✓ Final layer replaced (512 -> 2)

Model Parameters:
Total: 11,177,538
Trainable: 1,026 (0.01%)


In [9]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Print progress every 20 batches
        if (batch_idx + 1) % 20 == 0:
            print(f'  Batch [{batch_idx+1}/{len(dataloader)}] '
                  f'Loss: {running_loss/(batch_idx+1):.4f} '
                  f'Acc: {100.*correct/total:.2f}%')

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc


def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss = running_loss / len(dataloader)
    val_acc = 100. * correct / total

    return val_loss, val_acc

In [ ]:
def train_transfer_learning_model(model, train_loader, val_loader,
                                  epochs=5, lr=0.001):

    criterion = nn.CrossEntropyLoss()

    # Only optimize the final layer (since others are frozen)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                          lr=lr)

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

    # Track history
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    best_val_acc = 0.0

    print(f"\n{'='*60}")
    print(f"Training Transfer Learning Model")
    print(f"{'='*60}\n")

    for epoch in range(epochs):
        start_time = time.time()

        print(f"Epoch [{epoch+1}/{epochs}]")

        # Train
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        # Validate
        val_loss, val_acc = validate(
            model, val_loader, criterion, device
        )

        # Update scheduler
        scheduler.step()

        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Print epoch summary
        epoch_time = time.time() - start_time
        print(f'\n  Summary:')
        print(f'    Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'    Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        print(f'    Time: {epoch_time:.2f}s | LR: {scheduler.get_last_lr()[0]:.6f}')

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_transfer_model.pth')
            print(f'    ✓ Best model saved! (Val Acc: {val_acc:.2f}%)')

        print()

    print(f"{'='*60}")
    print(f"Training Complete! Best Val Accuracy: {best_val_acc:.2f}%")
    print(f"{'='*60}\n")

    return history

# Train the model
transfer_history = train_transfer_learning_model(
    transfer_model,
    train_loader,
    val_loader,
    epochs=5,
    lr=0.001  # Small learning rate for fine-tuning
)


Training Transfer Learning Model

Epoch [1/5]
  Batch [20/185] Loss: 0.4823 Acc: 80.00%
  Batch [40/185] Loss: 0.3700 Acc: 85.31%
  Batch [60/185] Loss: 0.3062 Acc: 88.70%
  Batch [80/185] Loss: 0.2684 Acc: 90.23%
  Batch [100/185] Loss: 0.2530 Acc: 90.78%
  Batch [120/185] Loss: 0.2345 Acc: 91.54%
  Batch [140/185] Loss: 0.2173 Acc: 92.21%
  Batch [160/185] Loss: 0.2043 Acc: 92.73%
  Batch [180/185] Loss: 0.1938 Acc: 93.18%

  Summary:
    Train Loss: 0.1911 | Train Acc: 93.30%
    Val Loss: 0.0696 | Val Acc: 97.50%
    Time: 56.95s | LR: 0.001000
    ✓ Best model saved! (Val Acc: 97.50%)

Epoch [2/5]
  Batch [20/185] Loss: 0.1106 Acc: 96.09%
  Batch [40/185] Loss: 0.1140 Acc: 96.25%
  Batch [60/185] Loss: 0.1092 Acc: 96.35%
  Batch [80/185] Loss: 0.1108 Acc: 96.29%
  Batch [100/185] Loss: 0.1085 Acc: 96.28%
  Batch [120/185] Loss: 0.1054 Acc: 96.43%
  Batch [140/185] Loss: 0.1047 Acc: 96.47%
  Batch [160/185] Loss: 0.1008 Acc: 96.58%
  Batch [180/185] Loss: 0.1006 Acc: 96.53%

  Sum

In [ ]:
def predict_image(model, image_path, transform):
    """Predict class for a single image."""
    model.eval()

    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        confidence, predicted = probabilities.max(1)

    classes = ['Cat', 'Dog']
    return classes[predicted.item()], confidence.item()


def visualize_predictions(model, dataset, num_images=8):
    """Visualize predictions on sample images."""
    model.eval()

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()

    # Get random samples
    indices = np.random.choice(len(dataset), num_images, replace=False)

    for idx, ax in zip(indices, axes):
        img_path, true_label = dataset.samples[idx]

        # Predict
        predicted_class, confidence = predict_image(model, img_path, val_transform)

        # Load image for display
        image = Image.open(img_path)

        # Display
        ax.imshow(image)
        ax.axis('off')

        true_class = dataset.classes[true_label].capitalize()
        color = 'green' if predicted_class.lower() == true_class.lower() else 'red'

        ax.set_title(f'True: {true_class}\nPred: {predicted_class} ({confidence*100:.1f}%)',
                    color=color, fontweight='bold')

    plt.tight_layout()
    plt.savefig('sample_predictions.png', dpi=300, bbox_inches='tight')
    plt.show()

# Visualize predictions
visualize_predictions(transfer_model, val_dataset)